# StormEngine V6 — End-to-End Adriatic Forecasting

This notebook launches the reproducible repository pipeline. It uses 239 coastal physical coordinates plus 151 virtual Adriatic sea support points, 2010–2015 training, 2016 validation, 2017 testing, train-only normalization, Natural Earth LSM, station-distance fields, and V6 mean-normalized sea-weighted MSE.

The training implementation lives in `scripts/train.py`, so checkpoints remain compatible between Windows CUDA and Mac CPU/MPS.

In [ ]:
from pathlib import Path
import json, subprocess, sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), 'Open this notebook from StormEngine-DL or its notebooks folder'
print('Repository:', REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO}[notebook]'], check=True)
import yaml

def run_live(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

## 1. Set the ERA5 folder on this computer
Change only `ERA5_ROOT`. It must contain the monthly files referenced by `data/manifests/era5_manifest.csv`. On the original layout it is the `DownloadDate` folder beside the repository.

In [ ]:
WINDOWS_ERA5_ROOT = Path(r'D:\Documents\py_projects\StormEngine-DL\DownloadDate')
ERA5_ROOT = WINDOWS_ERA5_ROOT if WINDOWS_ERA5_ROOT.exists() else (REPO.parent / 'DownloadDate').resolve()
BATCH_SIZE = 16  # RTX 4060 default; use 8 or 4 only if CUDA reports out-of-memory

config = yaml.safe_load((REPO / 'configs' / 'era5_2010_2017.yaml').read_text(encoding='utf-8'))
config['data']['era5_root'] = str(ERA5_ROOT)
config['training']['batch_size'] = BATCH_SIZE
config['training']['num_workers'] = 0
local_config = REPO / 'configs' / 'era5_2010_2017_windows.local.yaml'
local_config.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
pilot_config = yaml.safe_load((REPO / 'configs' / 'pilot.yaml').read_text(encoding='utf-8'))
pilot_config['data']['era5_root'] = str(ERA5_ROOT)
pilot_config['training']['batch_size'] = BATCH_SIZE
pilot_config['training']['num_workers'] = 0
pilot_local_config = REPO / 'configs' / 'pilot_windows.local.yaml'
pilot_local_config.write_text(yaml.safe_dump(pilot_config, sort_keys=False), encoding='utf-8')
print('ERA5:', ERA5_ROOT)
print('Full config:', local_config)
print('Pilot config:', pilot_local_config)

## 2. Install and verify CUDA
Install the CUDA-enabled PyTorch build on the Windows computer first if `torch.cuda.is_available()` is false. The command below installs this repository without rebuilding the already-versioned static fields.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training can be checked on CPU/MPS, but use the Windows CUDA computer for the full run.')

## 3. Build the one-time fast training cache
This converts the monthly NetCDF archive once into normalized memory-mapped arrays. Expect about 2 GB in `DownloadDate/cache/stormengine_2010_2017`. Every later run reuses it immediately.

In [ ]:
cache_dir = ERA5_ROOT / 'cache' / 'stormengine_2010_2017'
if (cache_dir / 'metadata.json').exists():
    print('Training cache already exists:', cache_dir)
else:
    run_live([sys.executable, '-u', str(REPO / 'scripts' / 'build_training_cache.py'), '--config', str(local_config)])

## 4. Data and model preflight
This must report `data source: memory-mapped cache`, then checks all three chronological splits and one complete 390-point model forward pass.

In [ ]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'check_data_pipeline.py'), '--config', str(local_config)])

## 5. Small smoke run
Run two training batches and one validation/test batch first. These values only verify the cached pipeline and are not scientific results.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
smoke_command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'train.py'), '--config', str(local_config),
    '--device', DEVICE, '--epochs', '1', '--max-train-batches', '2',
    '--max-eval-batches', '1', '--output-dir', 'artifacts/v6_smoke', '--evaluate-test'
]
run_live(smoke_command)

## 6. Medium 2010–2012 pilot
Run three short epochs (200 train batches and 50 validation batches) to verify learning direction, speed, and GPU memory before the full experiment. The held-out test set remains untouched.

In [ ]:
pilot_command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'train.py'), '--config', str(pilot_local_config),
    '--device', DEVICE, '--epochs', '3', '--max-train-batches', '200',
    '--max-eval-batches', '50', '--output-dir', 'artifacts/v6_pilot_check'
]
run_live(pilot_command)

## 7. Full 2010–2017 training or resume
Start this only after the pilot loss is finite and generally decreasing. Set `RESUME=True` after an interrupted run. Model selection uses only 2016; this command does not inspect 2017.

In [ ]:
RESUME = False
command = [sys.executable, '-u', str(REPO / 'scripts' / 'train.py'), '--config', str(local_config), '--device', DEVICE]
last_checkpoint = REPO / 'artifacts' / 'v6_2010_2017' / 'last.pt'
if RESUME:
    assert last_checkpoint.exists(), last_checkpoint
    command += ['--resume', str(last_checkpoint)]
run_live(command)

## 8. Evaluate the selected checkpoint on 2016 validation
This produces full/land/sea metrics for every lead hour without touching the held-out 2017 test set.

In [ ]:
artifact_dir = REPO / 'artifacts' / 'v6_2010_2017'
best_checkpoint = artifact_dir / 'best.pt'
run_live([
    sys.executable, '-u', str(REPO / 'scripts' / 'evaluate.py'), '--config', str(local_config),
    '--checkpoint', str(best_checkpoint), '--split', 'validation', '--device', DEVICE, '--save-examples', '3'
])

## 9. Inspect training curves
Use `StormEngine_V6_Evaluation.ipynb` for lead-hour plots, example maps, and the explicitly unlocked final test.

In [ ]:
import matplotlib.pyplot as plt
artifact_dir = REPO / 'artifacts' / 'v6_2010_2017'
history = json.loads((artifact_dir / 'history.json').read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [row['train_loss'] for row in history], label='train')
plt.plot(epochs, [row['validation_loss'] for row in history], label='validation')
plt.xlabel('Epoch'); plt.ylabel('V6 weighted MSE (normalized)'); plt.grid(alpha=.3); plt.legend(); plt.show()